# 14-day cumulative number of COVID-19 cases per 100 000 for districts (Landkreise) in Germany

At the end of the page, we provide a detailed description of how the numbers are calculated.

### Compute data

In [1]:
import datetime as dt
import pandas as pd
pd.set_option("max_rows", None)
from oscovida import get_incidence_rates_germany

## Table for all districts (=Landkreise)

In [2]:
cases_incidence, deaths_incidence = get_incidence_rates_germany()

Please be patient - downloading data from https://www.arcgis.com/sharing/rest/content/items/f10774f1c63e40168479a1feb6c7ca74/data ...
Completed downloading 232947 rows in 4.8 seconds.


In [3]:
cases_incidence

,14-day-sum,population,14-day-incidence-rate
Landkreis,,,
LK Ahrweiler,19,129727,14.6
LK Aichach-Friedberg,28,133596,21.0
LK Alb-Donau-Kreis,63,196047,32.1
LK Altenburger Land,6,90118,6.7
LK Altenkirchen,23,128705,17.9
LK Altmarkkreis Salzwedel,4,83765,4.8
LK Altötting,18,111210,16.2
LK Alzey-Worms,29,129244,22.4
LK Amberg-Sulzbach,21,103109,20.4


## Table sorted by 14-day-incidence

In [4]:
cases_incidence.sort_values(by="14-day-incidence-rate", ascending=False)

,14-day-sum,population,14-day-incidence-rate
Landkreis,,,
SK Hamm,330,179111,184.2
SK Remscheid,128,110994,115.3
LK Cloppenburg,170,169348,100.4
SK Berlin Mitte,369,384172,96.1
LK Rhön-Grabfeld,75,79690,94.1
LK Dingolfing-Landau,88,96217,91.5
SK Berlin Friedrichshain-Kreuzberg,262,289762,90.4
SK Berlin Neukölln,289,329691,87.7
SK München,1231,1471508,83.7


## Tutorial: Detailed calculation for one district

In [5]:
from oscovida import fetch_data_germany, germany_get_population
import datetime

In [6]:
period = 14 # Set the period we compute the incidence rate over

In [7]:
germany = fetch_data_germany() # Downloads population data for Germany
germany = germany.rename(
    columns={"AnzahlFall": "cases", "AnzahlTodesfall": "deaths"}
) # Rename the columns to English
germany.tail() # Print example of the table format and data

,FID,IdBundesland,Bundesland,Landkreis,Altersgruppe,Geschlecht,cases,deaths,Meldedatum,IdLandkreis,Datenstand,NeuerFall,NeuerTodesfall,Refdatum,NeuGenesen,AnzahlGenesen,IstErkrankungsbeginn,Altersgruppe2
date,,,,,,,,,,,,,,,,,,
2020-05-06,40025313,16,Thüringen,LK Altenburger Land,A80+,W,1,0,2020/05/06 00:00:00,16077,"01.10.2020, 00:00 Uhr",0,-9,2020/05/04 00:00:00,0,1,1,Nicht übermittelt
2020-05-11,40025314,16,Thüringen,LK Altenburger Land,A80+,W,1,0,2020/05/11 00:00:00,16077,"01.10.2020, 00:00 Uhr",0,-9,2020/05/01 00:00:00,0,1,1,Nicht übermittelt
2020-05-28,40025315,16,Thüringen,LK Altenburger Land,A80+,W,2,0,2020/05/28 00:00:00,16077,"01.10.2020, 00:00 Uhr",0,-9,2020/05/28 00:00:00,0,2,0,Nicht übermittelt
2020-06-09,40025316,16,Thüringen,LK Altenburger Land,A80+,W,1,0,2020/06/09 00:00:00,16077,"01.10.2020, 00:00 Uhr",0,-9,2020/06/09 00:00:00,0,1,0,Nicht übermittelt
2020-09-30,40025317,16,Thüringen,LK Altenburger Land,A80+,W,1,0,2020/09/30 00:00:00,16077,"01.10.2020, 00:00 Uhr",1,-9,2020/09/23 00:00:00,-9,0,1,Nicht übermittelt


In [8]:
yesterday = datetime.datetime.now() - datetime.timedelta(days=1)
fortnight_ago = yesterday - datetime.timedelta(days=period)
periods = (fortnight_ago < germany.index) & (germany.index < yesterday) # Create a period mask
germany = germany.iloc[periods]

In [9]:
cases_germany = germany[["Landkreis", "cases"]]
cases_germany.tail() # The data for germany has the date as the index, and the location data on each column

,Landkreis,cases
date,,
2020-09-24,LK Altenburger Land,1
2020-09-22,LK Altenburger Land,1
2020-09-22,LK Altenburger Land,1
2020-09-22,LK Altenburger Land,1
2020-09-30,LK Altenburger Land,1


In [10]:
cases_sum = (
    cases_germany.groupby("Landkreis")
    .sum()
    .rename(columns={"cases": f"{period}-day-sum"})
) # Here we sum the daily data to get total data
cases_sum.tail()

,14-day-sum
Landkreis,
SK Worms,22
SK Wuppertal,155
SK Würzburg,88
SK Zweibrücken,7
StadtRegion Aachen,204


In [11]:
population = (
    germany_get_population()
    .population
    .to_frame()
) # This gives us population per county, with county as the index
population.tail()

,population
county,
SK Berlin Spandau,243977
SK Berlin Steglitz-Zehlendorf,308697
SK Berlin Mitte,384172
SK Berlin Friedrichshain-Kreuzberg,289762
SK Berlin Tempelhof-Schöneberg,351644


In [12]:
cases_incidence = cases_sum.join(population)
cases_incidence[f"{period}-day-incidence-rate"] = (
    cases_incidence[f"{period}-day-sum"] / cases_incidence["population"] * 100_000
) # By convention the incedence rate is the total cases in the past x days / population * 100_000

In [13]:
cases_incidence.tail()

,14-day-sum,population,14-day-incidence-rate
Landkreis,,,
SK Worms,22,83330,26.401056
SK Wuppertal,155,354382,43.738113
SK Würzburg,88,127880,68.814514
SK Zweibrücken,7,34209,20.462451
StadtRegion Aachen,204,555465,36.725986


In [14]:
cases_incidence.loc['SK Hamburg']

14-day-sum               8.570000e+02
population               1.841179e+06
14-day-incidence-rate    4.654626e+01
Name: SK Hamburg, dtype: float64